# 05 — Top non-government, non-party advocacy spenders

**The question**: among advertisers that are *neither government nor a registered party/candidate*, who spent the most on political-adjacent FB ads around the May 2022 federal election, and how do their spending patterns differ across the campaign window?

**Why exclude parties and candidates**: party central offices and candidate ads dominate raw spend totals (Australian Labor Party + UAP alone account for ~$5M pre-election spend). Their election-only behaviour is expected and uninteresting. The structurally interesting question is who's *in the advocacy ecosystem alongside them* — NGO campaigns, PAC-style election funders, single-issue advocacy groups — and how that ecosystem's spending pattern compares.

**Headline deliverable**: ranked list of the top 20 non-government, non-party advocacy spenders, with pre/post-election spend, persistence ratios, and topic mix.

**Input corpus**: v3 parquet from [04_topic_join.ipynb](04_topic_join.ipynb), filtered down to `match_type IS NULL` (i.e. only the LDA-classified advocacy residual — no candidate, no party_org, no government).

**Window**: 6 months before to 6 months after the 21 May 2022 election.

## 1. Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as spark_sum, count as spark_count, countDistinct,
    when, lit, date_trunc, desc, mode, broadcast,
)
from pyspark.sql.window import Window
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

spark = SparkSession.builder \
    .appName('FB_API_election_spenders') \
    .config('spark.sql.parquet.output.committer.class',
            'org.apache.parquet.hadoop.ParquetOutputCommitter') \
    .config('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

In [ ]:
V3_PATH = '/user/s3348393/main/preprocessing/v3/parquet'

ELECTION_DATE = pd.Timestamp('2022-05-21')
WINDOW_START  = pd.Timestamp('2021-11-21')
WINDOW_END    = pd.Timestamp('2022-11-21')

# Consistent palette across every chart in this notebook so the same advertiser
# type reads the same colour everywhere.
GROUP_COLORS = {
    'candidate':           '#1f77b4',   # blue
    'party_org':           '#ff7f0e',   # orange
    'climate':             '#2ca02c',   # green
    'humanitarian_rights': '#d62728',   # red
    'political_advocacy':  '#9467bd',   # purple
    'cost_of_living':      '#8c564b',   # brown
}

## 2. Load v3 and restrict to advocacy

Two filters applied at load:

1. **`match_type IS NULL`** — drop candidate and party_org. We're explicitly looking at the advocacy ecosystem only.
2. **Tag a `group` column** = LDA `category` (`climate`, `humanitarian_rights`, `political_advocacy`, `cost_of_living`).

Cache the result — every subsequent cell scans it.

In [ ]:
v3 = spark.read.parquet(V3_PATH)

# Drop candidate and party_org — focus on advocacy ecosystem only.
v3 = v3.filter(col('match_type').isNull())

# Use category as the group label (climate / humanitarian_rights / political_advocacy / cost_of_living).
v3 = v3.withColumn('group', col('category'))

# Tag pre/post election by ad creation date.
v3 = v3.withColumn(
    'period',
    when(col('ad_creation_date') < lit('2022-05-21'), 'pre')
    .otherwise('post')
)

v3 = v3.cache()
print(f'Advocacy ads in v3: {v3.count():,}')
print('\nGroup distribution:')
v3.groupBy('group').agg(
    spark_count('*').alias('ads'),
    spark_sum('spend_mid').alias('total_spend'),
).orderBy(desc('total_spend')).show(truncate=False)

## 3. Top 20 election spenders (table)

Rank bylines by total spend during the **pre-election** window (Nov 2021 – May 2022) — this is the campaign-period money. For each, also compute post-election spend and a persistence ratio.

Persistence = `post_spend / pre_spend`. Bands:
- `persistence < 0.2` → **election-only** (PAC-style — barely visible after election)
- `0.2 ≤ persistence ≤ 0.8` → **transitional**
- `persistence > 0.8` → **ongoing advocate** (steady-state advertiser)

In [ ]:
# Per-byline pre and post spend totals.
byline_spend = (v3
    .filter(col('spend_mid').isNotNull() & col('bylines').isNotNull())
    .groupBy('bylines')
    .pivot('period', ['pre', 'post'])
    .agg(spark_sum('spend_mid'))
).na.fill(0)

# Modal group per byline — collect counts in Spark, pick max in pandas.
group_per_byline = (v3.filter(col('bylines').isNotNull())
    .groupBy('bylines', 'group').count()
    .toPandas())

primary_group = (group_per_byline
    .sort_values(['bylines', 'count'], ascending=[True, False])
    .drop_duplicates('bylines', keep='first')
    [['bylines', 'group']])

# Bring spend to pandas, merge group, compute persistence, sort.
spend_pdf = byline_spend.toPandas()
spend_pdf.columns = ['bylines', 'pre_spend', 'post_spend']
spend_pdf = spend_pdf.merge(primary_group, on='bylines', how='left')
spend_pdf['total_spend'] = spend_pdf['pre_spend'] + spend_pdf['post_spend']
spend_pdf['persistence'] = spend_pdf['post_spend'] / spend_pdf['pre_spend'].replace(0, pd.NA)
spend_pdf['band'] = pd.cut(
    spend_pdf['persistence'].fillna(0),
    bins=[-0.001, 0.2, 0.8, float('inf')],
    labels=['election_only', 'transitional', 'ongoing']
)

top20 = spend_pdf.sort_values('pre_spend', ascending=False).head(20).reset_index(drop=True)
top20[['bylines', 'group', 'pre_spend', 'post_spend', 'persistence', 'band']]

## 4. Absolute spend over time — top 20 spenders

Weekly `spend_mid` sum per byline, restricted to the top 20. Lines coloured by `group` so the four match-type / category families read as distinct colour bands. Election-day marker for orientation.

Election-only advertisers concentrate spending in a narrow pre-election spike; ongoing advocates spread spending across the year.

In [ ]:
top20_bylines = top20['bylines'].tolist()

weekly = (v3
    .filter(col('bylines').isin(top20_bylines) & col('spend_mid').isNotNull())
    .withColumn('week', date_trunc('week', 'ad_creation_date'))
    .groupBy('week', 'bylines', 'group')
    .agg(spark_sum('spend_mid').alias('spend'))
    .orderBy('week')
    .toPandas())

fig, ax = plt.subplots(figsize=(14, 7))
for byl, sub in weekly.groupby('bylines'):
    grp = sub['group'].iloc[0]
    ax.plot(sub['week'], sub['spend'],
            color=GROUP_COLORS.get(grp, '#777777'),
            alpha=0.75, linewidth=1.5,
            label=f'{byl[:35]} ({grp})')

ax.axvline(ELECTION_DATE, linestyle='--', color='red', alpha=0.7)
ax.text(ELECTION_DATE + pd.Timedelta(days=2), ax.get_ylim()[1] * 0.95,
        ' Election day', color='red', va='top')
ax.set_title('Weekly spend — top 20 non-government election spenders')
ax.set_xlabel('Week of ad creation')
ax.set_ylabel('Spend ($)')
ax.legend(loc='upper right', fontsize=7, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Proportional spend over time — who held the megaphone?

Same data, normalised to each week's total spend across the top 20. Stacked area to 100%. Shows compositional dominance — UAP grabs majority share for ~6 weeks of the campaign, then ongoing advocates collectively reclaim the field after election day.

In [ ]:
pivot = weekly.pivot_table(
    index='week', columns='bylines', values='spend', aggfunc='sum'
).fillna(0)
# Order columns so largest total spenders are at the bottom of the stack
col_order = pivot.sum().sort_values(ascending=False).index.tolist()
pivot = pivot[col_order]
# Proportional
prop = pivot.div(pivot.sum(axis=1), axis=0).fillna(0)

# Build per-byline colours from primary group
byline_group = dict(zip(top20['bylines'], top20['group']))
colors = [GROUP_COLORS.get(byline_group.get(b), '#777777') for b in prop.columns]

fig, ax = plt.subplots(figsize=(14, 7))
prop.plot.area(ax=ax, color=colors, alpha=0.7, linewidth=0)
ax.axvline(ELECTION_DATE, linestyle='--', color='red', alpha=0.9, linewidth=1.5)
ax.text(ELECTION_DATE + pd.Timedelta(days=2), 0.97, ' Election day',
        color='red', va='top')
ax.set_title('Proportional weekly spend — share within top 20')
ax.set_xlabel('Week')
ax.set_ylabel('Share')
ax.set_ylim(0, 1)
ax.legend(loc='upper right', fontsize=7, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Persistence scatter — election-only vs ongoing

Each top-20 byline as a point. X-axis = total spend (log scale). Y-axis = persistence ratio. Coloured by `group`. Quadrants:

- Top-left: high spend, low persistence → **election-only PAC-style**
- Top-right: high spend, high persistence → **ongoing advocate**
- Bottom-left: low spend, low persistence → one-off campaign
- Bottom-right: low spend, high persistence → small persistent advertiser

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

for grp, sub in top20.groupby('group'):
    ax.scatter(
        sub['total_spend'], sub['persistence'].fillna(0),
        s=120, alpha=0.7,
        color=GROUP_COLORS.get(grp, '#777777'),
        label=grp, edgecolors='black', linewidth=0.5,
    )
for _, row in top20.iterrows():
    ax.annotate(
        row['bylines'][:25],
        (row['total_spend'], row['persistence'] if pd.notna(row['persistence']) else 0),
        fontsize=7, alpha=0.85, xytext=(5, 5), textcoords='offset points',
    )

# Reference lines for the persistence bands
ax.axhline(0.2, linestyle=':', color='gray', alpha=0.6)
ax.axhline(0.8, linestyle=':', color='gray', alpha=0.6)
ax.text(ax.get_xlim()[1] * 0.95, 0.1, 'election-only',
        color='gray', ha='right', fontsize=8)
ax.text(ax.get_xlim()[1] * 0.95, 0.5, 'transitional',
        color='gray', ha='right', fontsize=8)
ax.text(ax.get_xlim()[1] * 0.95, 1.0, 'ongoing',
        color='gray', ha='right', fontsize=8)

ax.set_xscale('log')
ax.set_xlabel('Total spend, log scale ($)')
ax.set_ylabel('Persistence ratio (post / pre)')
ax.set_title('Top 20 non-government election spenders — persistence vs spend')
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Pre vs post spend — bar chart

Direct side-by-side comparison for each top-20 byline. Election-only advertisers have near-zero post bars; ongoing advocates have comparable pre/post bars.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
y = range(len(top20))
bar_height = 0.4
ax.barh([i - bar_height/2 for i in y], top20['pre_spend'],
        height=bar_height, label='pre-election', color='#1f77b4', alpha=0.85)
ax.barh([i + bar_height/2 for i in y], top20['post_spend'],
        height=bar_height, label='post-election', color='#d62728', alpha=0.85)

ax.set_yticks(list(y))
ax.set_yticklabels([b[:50] for b in top20['bylines']], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Spend ($)')
ax.set_title('Pre-election vs post-election spend — top 20')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Persistence band summary table

Roll the top-20 up to bands — count and total spend per band.

In [ ]:
band_summary = top20.groupby('band', observed=True).agg(
    n_advertisers=('bylines', 'count'),
    pre_spend=('pre_spend', 'sum'),
    post_spend=('post_spend', 'sum'),
    total_spend=('total_spend', 'sum'),
).reset_index()

band_summary['pre_share']  = band_summary['pre_spend']  / band_summary['pre_spend'].sum()
band_summary['post_share'] = band_summary['post_spend'] / band_summary['post_spend'].sum()

band_summary

## 9. Discussion

*(Filled in after running the analysis — sketching headline findings)*

**Three findings expected**:

1. **The advocacy ecosystem is itself bimodal.** Even after excluding parties and candidates, the remaining advertisers split cleanly into election-only (PAC-style) and ongoing (NGO-style) bands. Solutions for Australia, Smart Voting, Climate 200, Shell (greenwashing), Pharmacy Guild (Affordable Medicines Now) cluster in the election-only band — they exist or run their campaigns *because of* the election. Greenpeace, Amnesty, ACF, UNHCR run year-round.

2. **PAC-style spending is highly concentrated in a few entities.** A small handful of right-leaning advocacy organisations (Solutions for Australia, Advance Australia, Smart Voting) account for a large fraction of election-only advocacy spend, alongside a few left-leaning equivalents (Climate 200's federal independents campaign).

3. **Greenpeace is the clearest ongoing advocate.** Persistence ratio near 1.0, total spend in the top tier. Their year-round campaigning provides the baseline against which everyone else's election concentration is measurable.

**Main message**: the non-party, non-government advocacy ecosystem is structurally split between *campaign-time PAC-style spenders* (organisations whose ad activity is essentially gated by election cycles) and *steady-state NGO advocates* (organisations whose advocacy continues regardless of electoral timing). The split is observable empirically through persistence ratios and complements the more obvious bimodal pattern visible in party-funded advertising.